In [20]:
#!pip install numpy pandas scipy mne
#!pip install mne-connectivity
# !pip install openpyxl


import os
base_path = "datatxt"
print(os.listdir(base_path))
nfb_dul = os.listdir(os.path.join(base_path,"tdt files Baseline only NFB+DUL who completed T2"))
nfb = os.listdir(os.path.join(base_path,"tdt files Baseline Only NFB who completed T2"))
print()
print(nfb_dul[0], nfb[0])

txt_path = os.path.join(base_path,"tdt files Baseline Only NFB who completed T2", nfb[0])

['.DS_Store', 'all_features', 'tdt files Baseline only NFB+DUL who completed T2', 'tdt files Baseline Only NFB who completed T2']

CIPN3253_BL_EO.txt CIPN3123_BL_EO.txt


# Parallel Features Extraction

In [ ]:
# Process all TXT files in both folders in parallel
import glob
from multiprocessing.dummy import Pool as ThreadPool
from pathlib import Path

from eeg_feature_extraction import extract_fft_feature_tables

# Define output directory
output_dir = "processeddata"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

def process_single_file(args):
    """Process a single TXT file and save results with appropriate naming."""
    txt_path, folder_type = args
    
    try:
        # Extract base name (e.g., "CIPN3001" from "CIPN3001_BL_EO.txt")
        filename = os.path.basename(txt_path)
        base_name = filename.replace("_BL_EO.txt", "").replace(".txt", "")
        
        # Determine output suffix based on folder type
        if folder_type == "NFB_DUL":
            output_suffix = "_NFB_DUL"
        elif folder_type == "NFB":
            output_suffix = "_NFB"
        else:
            output_suffix = ""
        
        # Create output path in all_features directory (function will add _ALL_FEATURES.xlsx, so we'll rename it)
        out_prefix = os.path.join(output_dir, f"{base_name}{output_suffix}")
        
        # Process the file (this creates {base_name}{suffix}_ALL_FEATURES.xlsx)
        results = extract_fft_feature_tables(
            txt_path,
            out_prefix=out_prefix,
            win_sec_psd=2.0,
            conn_epoch_sec=8.0,
            drop_A1A2=True,
            output_one_excel=True,
        )
        
        # Rename the file to remove _ALL_FEATURES suffix
        old_file = f"{out_prefix}_ALL_FEATURES.xlsx"
        new_file = f"{out_prefix}.xlsx"
        
        if os.path.exists(old_file):
            if os.path.exists(new_file):
                os.remove(new_file)  # Remove if exists
            os.rename(old_file, new_file)
            return f"✓ Success: {filename} -> {new_file}"
        else:
            return f"⚠ Warning: {filename} processed but output file not found"
    
    except Exception as e:
        return f"✗ Error processing {os.path.basename(txt_path)}: {str(e)}"

# Find all TXT files in both folders
nfb_dul_folder = os.path.join(base_path, "tdt files Baseline only NFB+DUL who completed T2")
nfb_folder = os.path.join(base_path, "tdt files Baseline Only NFB who completed T2")

# Get all txt files with their folder types
files_to_process = []

# Files from NFB_DUL folder
for txt_file in glob.glob(os.path.join(nfb_dul_folder, "*.txt")):
    files_to_process.append((txt_file, "NFB_DUL"))

# Files from NFB folder
for txt_file in glob.glob(os.path.join(nfb_folder, "*.txt")):
    files_to_process.append((txt_file, "NFB"))

print(f"Found {len(files_to_process)} files to process")
print(f"NFB_DUL files: {sum(1 for _, ftype in files_to_process if ftype == 'NFB_DUL')}")
print(f"NFB files: {sum(1 for _, ftype in files_to_process if ftype == 'NFB')}")
print(f"Output directory: {output_dir}")
print()

# Process files in parallel using threads (notebook-safe)
if files_to_process:
    with ThreadPool(processes=5) as pool:
        results = pool.map(process_single_file, files_to_process)
    
    # Print results
    for result in results:
        print(result)
    
    print(f"\n✓ Completed processing {len(files_to_process)} files")
else:
    print("No files found to process")


python: /Users/leoqian/mdanderson/EEGOutcomePrediction/venv/bin/python
mne_connectivity: 0.7.0 /Users/leoqian/mdanderson/EEGOutcomePrediction/venv/lib/python3.11/site-packages/mne_connectivity/__init__.py
eeg_feature_extraction reloaded
Found 78 files to process
NFB_DUL files: 21
NFB files: 57
Output directory: datatxt/all_features

Saved: datatxt/all_features/CIPN3013_NFB_DUL_ALL_FEATURES.xlsx
Saved: datatxt/all_features/CIPN3142_NFB_DUL_ALL_FEATURES.xlsx
Saved: datatxt/all_features/CIPN3002_NFB_DUL_ALL_FEATURES.xlsx
Saved: datatxt/all_features/CIPN3035_NFB_DUL_ALL_FEATURES.xlsx
Saved: datatxt/all_features/CIPN3253_NFB_DUL_ALL_FEATURES.xlsx
Saved: datatxt/all_features/CIPN3111_NFB_DUL_ALL_FEATURES.xlsx
Saved: datatxt/all_features/CIPN3180_NFB_DUL_ALL_FEATURES.xlsx
Saved: datatxt/all_features/CIPN3084_NFB_DUL_ALL_FEATURES.xlsx
Saved: datatxt/all_features/CIPN3109_NFB_DUL_ALL_FEATURES.xlsx
Saved: datatxt/all_features/CIPN3044_NFB_DUL_ALL_FEATURES.xlsx
Saved: datatxt/all_features/CIPN303

In [32]:
# Count processed feature files check if all patients are processed
import glob

output_dir = "processeddata"

processed_dir = output_dir
xlsx_files = glob.glob(os.path.join(processed_dir, "*.xlsx"))
print(f"Processed files in {processed_dir}: {len(xlsx_files)}")

Processed files in processeddata: 78


# Load 1 Example

In [29]:
# Load an Excel file and access a specific sheet
import pandas as pd

# Example: Load a specific Excel file
excel_file = os.path.join(output_dir, "CIPN3001_NFB.xlsx")  # Change to your desired file

# Method 1: List all available sheets
xl_file = pd.ExcelFile(excel_file)
print("Available sheets:")
for sheet_name in xl_file.sheet_names:
    print(f"  - {sheet_name}")

print("\n" + "="*50 + "\n")

# Method 2: Load a specific sheet by name
sheet_name = "FFT_abs_bandpower_uV2"  # Change to your desired sheet name
df = pd.read_excel(excel_file, sheet_name=sheet_name)
print(f"Loaded sheet: {sheet_name}")
print(f"Shape: {df.shape}")
print("\nFirst few rows:")
print(df.head())

# Method 3: Load multiple sheets at once
sheets_to_load = ["FFT_abs_bandpower_uV2", "FFT_rel_bandpower_pct", "PeakFreq_Hz"]
data_dict = {}
for sheet in sheets_to_load:
    data_dict[sheet] = pd.read_excel(excel_file, sheet_name=sheet)

print(f"\nLoaded {len(data_dict)} sheets into dictionary")
print(f"Keys: {list(data_dict.keys())}")

# Method 4: Load all sheets at once
all_sheets = pd.read_excel(excel_file, sheet_name=None)  # Returns a dictionary of all sheets
print(f"\nTotal sheets loaded: {len(all_sheets)}")


Available sheets:
  - meta
  - FFT_abs_bandpower_uV2
  - Z_FFT_abs_bandpower_uV2
  - FFT_rel_bandpower_pct
  - Z_FFT_rel_bandpower_pct
  - FFT_abs_1to50Hz_uV2
  - Z_FFT_abs_1to30Hz_uV2
  - Z_FFT_abs_1to50Hz_uV2
  - FFT_rel_1to50Hz_pct
  - Z_FFT_rel_1to30Hz_pct
  - Z_FFT_rel_1to50Hz_pct
  - PeakFreq_Hz
  - Z_PeakFreq_Hz
  - FFT_Coherence
  - Z_FFT_Coherence
  - FFT_PhaseLag_PLI
  - Z_FFT_PhaseLag_PLI


Loaded sheet: FFT_abs_bandpower_uV2
Shape: (19, 15)

First few rows:
  Channel         Delta        Theta       Alpha        Beta   HighBeta  \
0  FP1-LE     38.238512     4.128686    2.315065   28.494356  10.593170   
1  FP2-LE     23.345379     2.804118    2.133777   28.062195   9.817158   
2   F7-LE     19.260395     6.789990   10.565571   32.674180  15.812257   
3   F3-LE  11654.670171  1369.431272  352.111938  301.789499  30.561157   
4   Fz-LE      6.178930     0.761850    0.603269    2.371195   0.748935   

       Gamma  HighGamma      Alpha1      Alpha2       Beta1      Beta2  \
0